In [1]:
# -*- coding: utf-8 -*-
"""
SKRIP FINAL PENELITIAN: RANDOM FOREST 3 KELAS
Metodologi:
  1. Data Preparation + Encoding
  2. K-Fold Stratified Cross Validation (k=5)
  3. Feature Selection via RF Feature Importance
  5. Hyperparameter Tuning: GridSearchCV
  6. Evaluasi: F1, Precision, Recall, Accuracy, MCC | MAE, RMSE
  Skenario: Standard Scaler vs No Scaler
"""

import pandas as pd
import numpy as np
import shap
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    matthews_corrcoef, classification_report,
    mean_absolute_error, mean_squared_error, r2_score,
    confusion_matrix
)

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ============================================================
# LANGKAH 1: BACA & PERSIAPAN DATA
# ============================================================
print('=' * 65)
print('LANGKAH 1: PERSIAPAN DAN PEMBERSIHAN DATA')
print('=' * 65)

df = pd.read_csv('data_advanced.csv')
print(f'Shape awal: {df.shape}')

# Drop kolom identitas & data leakage
cols_to_drop = ['nisn', 'nama_siswa', 'email', 'gender', 'nilai_raport_zscore_jurusan']
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns], errors='ignore')
target_col = 'nilai_rata_rata_raport'

# ── Cek missing values ──
missing = df.isnull().sum()
if missing.sum() > 0:
    print(f'\nMissing values ditemukan:')
    print(missing[missing > 0])
    for col in df.columns:
        if df[col].isnull().any():
            if df[col].dtype == 'object':
                df[col].fillna(df[col].mode()[0], inplace=True)
                print(f'  [{col}] diisi modus: {df[col].mode()[0]}')
            else:
                df[col].fillna(df[col].median(), inplace=True)
                print(f'  [{col}] diisi median: {df[col].median():.2f}')
else:
    print('  ✓ Tidak ada missing values')

# ── Winsorize outlier (1%-99%) ──
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
if target_col in numeric_cols:
    numeric_cols.remove(target_col)

for col in numeric_cols:
    lo, hi = df[col].quantile(0.01), df[col].quantile(0.99)
    df[col] = np.clip(df[col], lo, hi)

# ── Enkoding Ordinal ──
ordinal_mappings = {
    'study_environment'    : ['Kurang Kondusif', 'Cukup Kondusif', 'Kondusif'],
    'kompetensi_skill_level': ['Rendah', 'Menengah', 'Tinggi'],
    'stress_level'         : ['Rendah', 'Sedang', 'Berat']
}
for col, cats in ordinal_mappings.items():
    if col in df.columns:
        oe = OrdinalEncoder(categories=[cats], dtype=int)
        df[col] = oe.fit_transform(df[[col]])

# ── Enkoding Nominal (OneHot) ──
nominal_cols        = ['kerja_sampingan', 'industry_readiness', 'jurusan']
nominal_cols_exist  = [c for c in nominal_cols if c in df.columns]
if nominal_cols_exist:
    onehot_encoder  = OneHotEncoder(sparse_output=False, drop=None)
    ohe_data        = onehot_encoder.fit_transform(df[nominal_cols_exist])
    ohe_names       = onehot_encoder.get_feature_names_out(nominal_cols_exist)
    df_onehot       = pd.DataFrame(ohe_data, columns=ohe_names, index=df.index)
else:
    df_onehot = pd.DataFrame(index=df.index)

df_ordinal   = df[[c for c in ordinal_mappings.keys() if c in df.columns]].copy()
X_all        = pd.concat([df[numeric_cols], df_ordinal, df_onehot], axis=1)
feature_names = X_all.columns.tolist()
y_reg         = df[target_col].values

print(f'\nShape setelah preprocessing: {X_all.shape}')
print(f'Jumlah fitur  : {len(feature_names)}')
print(f'Target range  : {y_reg.min():.2f} – {y_reg.max():.2f}  |  std: {y_reg.std():.2f}')


c:\Users\syauq\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LANGKAH 1: PERSIAPAN DAN PEMBERSIHAN DATA
Shape awal: (199, 20)
  ✓ Tidak ada missing values

Shape setelah preprocessing: (199, 19)
Jumlah fitur  : 19
Target range  : 75.60 – 88.33  |  std: 1.93


In [2]:
# ============================================================
# LANGKAH 2: TARGET ENGINEERING — 3 KELAS (Q25 / Q75)
# ============================================================
# Menggunakan Q25 dan Q75 agar jarak antar kelas lebih bermakna
# dibandingkan Q33/Q66 yang selisihnya hanya ~1.5 poin
print('\n' + '='*65)
print('LANGKAH 2: TARGET ENGINEERING')
print('='*65)

low_threshold  = df[target_col].quantile(0.25)
high_threshold = df[target_col].quantile(0.75)

def get_custom_class(score):
    if score <= low_threshold:
        return 'Beresiko'
    elif score <= high_threshold:
        return 'Aman'
    else:
        return 'Sangat Aman'

y_3class = np.array([get_custom_class(s) for s in df[target_col]])

dist = dict(pd.Series(y_3class).value_counts())
print(f'Threshold Beresiko   : nilai_rata_rata_raport <= {low_threshold:.2f}')
print(f'Threshold Aman       : {low_threshold:.2f} < nilai <= {high_threshold:.2f}')
print(f'Threshold Sangat Aman: nilai_rata_rata_raport  > {high_threshold:.2f}')
print(f'\nDistribusi kelas : {dist}')
print(f'Total sampel     : {sum(dist.values())}')



LANGKAH 2: TARGET ENGINEERING
Threshold Beresiko   : nilai_rata_rata_raport <= 82.90
Threshold Aman       : 82.90 < nilai <= 85.50
Threshold Sangat Aman: nilai_rata_rata_raport  > 85.50

Distribusi kelas : {'Aman': np.int64(102), 'Beresiko': np.int64(50), 'Sangat Aman': np.int64(47)}
Total sampel     : 199


In [3]:
# ============================================================
# LANGKAH 3: SELEKSI FITUR via RF FEATURE IMPORTANCE
# ============================================================
# Latih RF dasar di seluruh data → ambil fitur importance
# → buang fitur dengan importance sangat rendah / nol
print('\n' + '='*65)
print('LANGKAH 3: SELEKSI FITUR (Variable Screening)')
print('='*65)

# RF awal dengan semua fitur untuk mengukur importance
rf_screen = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf_screen.fit(X_all, y_3class)

# Tabel importance
importance_df = pd.DataFrame({
    'Fitur'     : feature_names,
    'Importance': rf_screen.feature_importances_
}).sort_values('Importance', ascending=False).reset_index(drop=True)

print('\nFeature Importance (seluruh fitur):')
print(importance_df.to_string(index=False))

# ── Seleksi: ambil fitur dengan importance > threshold ──
# Threshold = rata-rata importance (buang yang di bawah rata-rata)
importance_threshold = importance_df['Importance'].mean()
selected_mask        = rf_screen.feature_importances_ >= importance_threshold
selected_features    = np.array(feature_names)[selected_mask]
X                    = X_all[selected_features]    # DataFrame final

print(f'\nThreshold importance : {importance_threshold:.4f} (nilai rata-rata)')
print(f'Fitur terpilih ({len(selected_features)}) : {list(selected_features)}')
print(f'Fitur dibuang  ({len(feature_names) - len(selected_features)}) : '
      f'{[f for f in feature_names if f not in selected_features]}')



LANGKAH 3: SELEKSI FITUR (Variable Screening)

Feature Importance (seluruh fitur):
                        Fitur  Importance
         presentase_kehadiran    0.190471
                    ses_index    0.131762
                  screen_time    0.114613
         jam_belajar_per_hari    0.098487
         skor_time_management    0.078811
            motivasi_akademik    0.060237
            study_environment    0.050915
                 jurusan_TPTU    0.038658
                 stress_level    0.037651
                  jurusan_TKJ    0.037509
           jurusan_MULTIMEDIA    0.036057
        kerja_sampingan_Tidak    0.030344
           kerja_sampingan_Ya    0.028844
                  jurusan_TAV    0.021022
      industry_readiness_Siap    0.020392
industry_readiness_Belum Siap    0.019879
                    jam_tidur    0.002477
                deviasi_tidur    0.001871
       kompetensi_skill_level    0.000000

Threshold importance : 0.0526 (nilai rata-rata)
Fitur terpilih (6) : [np.st

In [4]:
# ============================================================
# LANGKAH 4 & 5: FUNGSI PELATIHAN & EVALUASI (K-FOLD CV)
# ============================================================

def run_scenario(X, y_cls, y_reg, use_scaler, scenario_name, n_splits=5):
    print(f'\nMenjalankan: {scenario_name} ...')
    print(f'  K-Fold: {n_splits}-fold Stratified CV')

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)

    # Container akumulasi metrik tiap fold
    fold_acc, fold_f1, fold_prec, fold_rec, fold_mcc = [], [], [], [], []
    fold_mae, fold_rmse, fold_r2                      = [], [], []

    best_fold_f1     = -1
    best_fold_model  = None
    best_fold_scaler = None
    best_X_test_fold = None
    best_params      = None

    # Grid hyperparameter — konservatif untuk data kecil
    param_grid_cls = {
        'n_estimators'     : [50, 100, 200],
        'max_depth'        : [3, 5, 7],
        'min_samples_split': [5, 10],
        'min_samples_leaf' : [2, 4],
        'max_features'     : ['sqrt', 'log2']
    }
    param_grid_reg = {
        'n_estimators'     : [50, 100, 200],
        'max_depth'        : [3, 5, 7],
        'min_samples_split': [5, 10],
        'min_samples_leaf' : [2, 4]
    }

    for fold_num, (train_idx, test_idx) in enumerate(skf.split(X, y_cls), 1):
        X_train_f = X.iloc[train_idx].values
        X_test_f  = X.iloc[test_idx].values
        y_train_c = y_cls[train_idx]
        y_test_c  = y_cls[test_idx]
        y_train_r = y_reg[train_idx]
        y_test_r  = y_reg[test_idx]

        # ── Scaling (fit HANYA di train, transform di test) ──
        if use_scaler:
            sc        = StandardScaler()
            X_train_f = sc.fit_transform(X_train_f)
            X_test_f  = sc.transform(X_test_f)
        else:
            sc = None

        # ── Klasifikasi: class_weight='balanced' menangani imbalance ──
        # (tanpa SMOTE / ADASYN)
        rf_cls = RandomForestClassifier(
            criterion='entropy',
            class_weight='balanced',
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
        grid_cls = GridSearchCV(
            rf_cls, param_grid_cls,
            cv=3, scoring='f1_macro', n_jobs=-1
        )
        grid_cls.fit(X_train_f, y_train_c)
        best_cls   = grid_cls.best_estimator_
        y_pred_cls = best_cls.predict(X_test_f)

        # ── Regresi ──
        rf_reg   = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1)
        grid_reg = GridSearchCV(
            rf_reg, param_grid_reg,
            cv=3, scoring='neg_mean_absolute_error', n_jobs=-1
        )
        grid_reg.fit(X_train_f, y_train_r)
        y_pred_reg = grid_reg.best_estimator_.predict(X_test_f)

        # ── Metrics fold ke-i ──
        acc  = accuracy_score(y_test_c, y_pred_cls)
        f1   = f1_score(y_test_c, y_pred_cls, average='macro', zero_division=0)
        prec = precision_score(y_test_c, y_pred_cls, average='macro', zero_division=0)
        rec  = recall_score(y_test_c, y_pred_cls, average='macro', zero_division=0)
        mcc  = matthews_corrcoef(y_test_c, y_pred_cls)
        mae  = mean_absolute_error(y_test_r, y_pred_reg)
        rmse = np.sqrt(mean_squared_error(y_test_r, y_pred_reg))
        r2   = r2_score(y_test_r, y_pred_reg)

        fold_acc.append(acc);   fold_f1.append(f1);   fold_prec.append(prec)
        fold_rec.append(rec);   fold_mcc.append(mcc)
        fold_mae.append(mae);   fold_rmse.append(rmse); fold_r2.append(r2)

        print(f'  Fold {fold_num}: Acc={acc:.3f}  F1={f1:.3f}  '
              f'Prec={prec:.3f}  Rec={rec:.3f}  MCC={mcc:.3f}  '
              f'|  MAE={mae:.3f}  RMSE={rmse:.3f}  R2={r2:.3f}')

        # Simpan model fold terbaik (berdasarkan F1)
        if f1 > best_fold_f1:
            best_fold_f1     = f1
            best_fold_model  = best_cls
            best_fold_scaler = sc
            best_X_test_fold = X_test_f
            best_params      = grid_cls.best_params_

    # ── Rata-rata semua fold ──
    def m(lst): return np.mean(lst), np.std(lst)

    acc_m,  acc_s  = m(fold_acc)
    f1_m,   f1_s   = m(fold_f1)
    prec_m, prec_s = m(fold_prec)
    rec_m,  rec_s  = m(fold_rec)
    mcc_m,  mcc_s  = m(fold_mcc)
    mae_m,  mae_s  = m(fold_mae)
    rmse_m, rmse_s = m(fold_rmse)
    r2_m,   r2_s   = m(fold_r2)

    print(f'  ─────────────────────────────────────────────────────────────────')
    print(f'  RATA²  : Acc={acc_m:.3f}±{acc_s:.3f}  F1={f1_m:.3f}±{f1_s:.3f}  '
          f'Prec={prec_m:.3f}±{prec_s:.3f}  Rec={rec_m:.3f}±{rec_s:.3f}  '
          f'MCC={mcc_m:.3f}±{mcc_s:.3f}')
    print(f'           MAE={mae_m:.3f}±{mae_s:.3f}  '
          f'RMSE={rmse_m:.3f}±{rmse_s:.3f}  R2={r2_m:.3f}±{r2_s:.3f}')
    print(f'  Best hyperparams: {best_params}')

    # ── SHAP dari model fold terbaik ──
    explainer    = shap.TreeExplainer(best_fold_model)
    shap_vals    = explainer.shap_values(best_X_test_fold)
    classes_list = list(best_fold_model.classes_)
    t_idx        = classes_list.index('Beresiko') if 'Beresiko' in classes_list else 0

    shap_target = shap_vals[t_idx] if isinstance(shap_vals, list) else (
                  shap_vals[:, :, t_idx] if len(shap_vals.shape) == 3 else shap_vals
    )
    shap_imp = np.abs(shap_target).mean(axis=0)
    imp_df   = pd.DataFrame(
        {'feature': selected_features, 'importance': shap_imp}
    ).sort_values('importance', ascending=False)
    top3 = ', '.join(imp_df.head(3)['feature'].tolist())

    return {
        'Skenario'         : scenario_name,
        'Accuracy'         : f'{acc_m:.4f} ± {acc_s:.4f}',
        'F1-Macro'         : f'{f1_m:.4f} ± {f1_s:.4f}',
        'Precision'        : f'{prec_m:.4f} ± {prec_s:.4f}',
        'Recall'           : f'{rec_m:.4f} ± {rec_s:.4f}',
        'MCC'              : f'{mcc_m:.4f} ± {mcc_s:.4f}',
        'MAE'              : f'{mae_m:.4f} ± {mae_s:.4f}',
        'RMSE'             : f'{rmse_m:.4f} ± {rmse_s:.4f}',
        'R2'               : f'{r2_m:.4f} ± {r2_s:.4f}',
        'Top 3 SHAP'       : top3,
        '_f1_float'        : f1_m,
        '_mae_float'       : mae_m,
        'model'            : best_fold_model,
        'scaler'           : best_fold_scaler,
        'selected_features': selected_features
    }


In [5]:
# ============================================================
# LANGKAH 5: EKSEKUSI DUA SKENARIO
# ============================================================
print('\n' + '='*65)
print('EKSEKUSI SKENARIO: SCALER VS NO SCALER')
print('='*65)

scenarios = [
    (True,  '1. Scaler + 3 Class'),
    (False, '2. No Scaler + 3 Class'),
]

results = []
for use_sc, name in scenarios:
    res = run_scenario(X, y_3class, y_reg, use_scaler=use_sc, scenario_name=name, n_splits=5)
    results.append(res)



EKSEKUSI SKENARIO: SCALER VS NO SCALER

Menjalankan: 1. Scaler + 3 Class ...
  K-Fold: 5-fold Stratified CV
  Fold 1: Acc=0.400  F1=0.400  Prec=0.417  Rec=0.426  MCC=0.122  |  MAE=1.335  RMSE=1.656  R2=0.038
  Fold 2: Acc=0.400  F1=0.309  Prec=0.308  Rec=0.310  MCC=0.007  |  MAE=1.527  RMSE=1.930  R2=-0.121
  Fold 3: Acc=0.425  F1=0.374  Prec=0.368  Rec=0.400  MCC=0.072  |  MAE=1.242  RMSE=1.568  R2=0.195
  Fold 4: Acc=0.550  F1=0.521  Prec=0.573  Rec=0.500  MCC=0.269  |  MAE=1.440  RMSE=1.831  R2=0.157
  Fold 5: Acc=0.308  F1=0.236  Prec=0.227  Rec=0.267  MCC=-0.099  |  MAE=1.755  RMSE=2.365  R2=-0.052
  ─────────────────────────────────────────────────────────────────
  RATA²  : Acc=0.417±0.078  F1=0.368±0.095  Prec=0.379±0.116  Rec=0.381±0.083  MCC=0.074±0.122
           MAE=1.460±0.176  RMSE=1.870±0.278  R2=0.043±0.120
  Best hyperparams: {'max_depth': 3, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 50}

Menjalankan: 2. No Scaler + 3 Class

In [6]:
# ============================================================
# LANGKAH 6: TABEL HASIL AKHIR PENELITIAN
# ============================================================
print('\n' + '='*80)
print('=== HASIL AKHIR PENELITIAN: 3 KELAS (SCALER VS NO SCALER) ===')
print('='*80)

best_result = max(results, key=lambda x: x['_f1_float'])
print(f"\n>> Model Terbaik: {best_result['Skenario']}  "
      f"(F1-Macro: {best_result['F1-Macro']})\n")

display_cols = ['Skenario','Accuracy','F1-Macro','Precision','Recall','MCC',
                'MAE','RMSE','R2','Top 3 SHAP']
summary_df = pd.DataFrame([{k: v for k, v in r.items() if k in display_cols}
                            for r in results])

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1200)
pd.set_option('display.max_colwidth', 60)
print(summary_df.to_string(index=False))
print('='*80)



=== HASIL AKHIR PENELITIAN: 3 KELAS (SCALER VS NO SCALER) ===

>> Model Terbaik: 2. No Scaler + 3 Class  (F1-Macro: 0.3775 ± 0.1099)

              Skenario        Accuracy        F1-Macro       Precision          Recall             MCC             MAE            RMSE              R2                                                       Top 3 SHAP
   1. Scaler + 3 Class 0.4165 ± 0.0778 0.3679 ± 0.0954 0.3787 ± 0.1161 0.3805 ± 0.0832 0.0743 ± 0.1221 1.4598 ± 0.1763 1.8700 ± 0.2783 0.0432 ± 0.1201 skor_time_management, presentase_kehadiran, jam_belajar_per_hari
2. No Scaler + 3 Class 0.4115 ± 0.1004 0.3775 ± 0.1099 0.3945 ± 0.1215 0.3852 ± 0.1006 0.0823 ± 0.1428 1.4599 ± 0.1740 1.8676 ± 0.2770 0.0460 ± 0.1160 skor_time_management, presentase_kehadiran, jam_belajar_per_hari


In [7]:
# ============================================================
# LANGKAH 6: VISUALISASI
# ============================================================
print('\nMembuat visualisasi...')
sns.set_theme(style='whitegrid')
plt.rcParams['font.family'] = 'sans-serif'

# ── Ekstrak nilai mean dari string 'x.xxxx ± y.yyyy' ──
def extract_mean(val_str):
    return float(val_str.split('±')[0].strip())

names   = [r['Skenario'] for r in results]
acc_v   = [extract_mean(r['Accuracy'])  for r in results]
f1_v    = [extract_mean(r['F1-Macro'])  for r in results]
prec_v  = [extract_mean(r['Precision']) for r in results]
rec_v   = [extract_mean(r['Recall'])    for r in results]
mcc_v   = [extract_mean(r['MCC'])       for r in results]
mae_v   = [extract_mean(r['MAE'])       for r in results]
rmse_v  = [extract_mean(r['RMSE'])      for r in results]

x = np.arange(len(names))
w = 0.15

# ── FIGURE 1: Metrik Klasifikasi ──
fig, ax = plt.subplots(figsize=(12, 6))
bars = [
    ax.bar(x - 2*w, acc_v,  w, label='Accuracy',  color='#4C72B0'),
    ax.bar(x -   w, f1_v,   w, label='F1-Macro',  color='#55A868'),
    ax.bar(x,       prec_v, w, label='Precision',  color='#C44E52'),
    ax.bar(x +   w, rec_v,  w, label='Recall',     color='#8172B2'),
    ax.bar(x + 2*w, mcc_v,  w, label='MCC',        color='#CCB974'),
]
for bar_group in bars:
    for rect in bar_group:
        h = rect.get_height()
        ax.annotate(f'{h:.3f}',
                    xy=(rect.get_x() + rect.get_width()/2, h),
                    xytext=(0, 3), textcoords='offset points',
                    ha='center', va='bottom', fontsize=8)
ax.set_title('Perbandingan Metrik Klasifikasi (5-Fold CV, Mean)', fontsize=14, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(names, fontsize=11)
ax.set_ylabel('Skor'); ax.legend(ncol=5, loc='upper right')
plt.tight_layout()
plt.savefig('grafik_klasifikasi_3class.png', dpi=300)
plt.close()

# ── FIGURE 2: Metrik Regresi ──
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].bar(names, mae_v,  color=['#4C72B0','#C44E52'])
for i, v in enumerate(mae_v): axes[0].text(i, v+0.01, f'{v:.3f}', ha='center', fontweight='bold')
axes[0].set_title('MAE (lebih rendah = lebih baik)', fontweight='bold')
axes[0].set_ylabel('MAE')

axes[1].bar(names, rmse_v, color=['#55A868','#8172B2'])
for i, v in enumerate(rmse_v): axes[1].text(i, v+0.01, f'{v:.3f}', ha='center', fontweight='bold')
axes[1].set_title('RMSE (lebih rendah = lebih baik)', fontweight='bold')
axes[1].set_ylabel('RMSE')

plt.suptitle('Perbandingan Metrik Regresi (5-Fold CV, Mean)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('grafik_regresi_3class.png', dpi=300)
plt.close()

# ── FIGURE 3: SHAP per kelas dari model terbaik ──
print('\nMembuat visualisasi SHAP per kelas dari model terbaik...')
model_best   = best_result['model']
scaler_best  = best_result['scaler']
sel_feats    = best_result['selected_features']

# Re-buat satu test set representatif untuk SHAP
from sklearn.model_selection import train_test_split
_, X_shap_raw, _, _ = train_test_split(
    X[sel_feats], y_3class, test_size=0.20, random_state=RANDOM_STATE, stratify=y_3class
)
X_shap = scaler_best.transform(X_shap_raw.values) if scaler_best else X_shap_raw.values

explainer   = shap.TreeExplainer(model_best)
shap_values = explainer.shap_values(X_shap)
classes_out = list(model_best.classes_)

fig, axes = plt.subplots(1, len(classes_out), figsize=(6*len(classes_out), 7))
if len(classes_out) == 1: axes = [axes]
fig.suptitle(f'SHAP Feature Importance per Kelas\n{best_result["Skenario"]}',
             fontsize=15, fontweight='bold', y=1.02)

for idx, cls_name in enumerate(classes_out):
    ax = axes[idx]
    sv = shap_values[idx] if isinstance(shap_values, list) else (
         shap_values[:, :, idx] if len(shap_values.shape) == 3 else shap_values
    )
    si  = np.abs(sv).mean(axis=0)
    pdf = pd.DataFrame({'Feature': sel_feats, 'Importance': si}
          ).sort_values('Importance', ascending=True).tail(8)
    ax.barh(pdf['Feature'], pdf['Importance'], color='#4C72B0', edgecolor='black')
    ax.set_title(f'Kelas: {cls_name.upper()}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Mean |SHAP|')
    ax.xaxis.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.savefig('grafik_shap_per_kelas_3class.png', dpi=300, bbox_inches='tight')
plt.close()

print('\nSemua visualisasi berhasil disimpan.')



Membuat visualisasi...

Membuat visualisasi SHAP per kelas dari model terbaik...

Semua visualisasi berhasil disimpan.


In [8]:
# ============================================================
# EXPORT MODEL TERBAIK
# ============================================================
import joblib

print('\nMenyimpan model terbaik dan transformer...')

joblib.dump(best_result['model'],  'model_rf_3class.pkl')
print('  ✓ model_rf_3class.pkl')

if best_result['scaler'] is not None:
    joblib.dump(best_result['scaler'], 'scaler.pkl')
    print('  ✓ scaler.pkl')

# Simpan OneHotEncoder
if nominal_cols_exist:
    joblib.dump(onehot_encoder, 'onehot_encoder.pkl')
    print('  ✓ onehot_encoder.pkl')

# Simpan OrdinalEncoders (re-fit)
ordinal_encoders_fitted = {}
for col, cats in ordinal_mappings.items():
    if col in df.columns:
        oe = OrdinalEncoder(categories=[cats], dtype=int)
        oe.categories_    = [np.array(cats)]
        oe.n_features_in_ = 1
        ordinal_encoders_fitted[col] = oe
joblib.dump(ordinal_encoders_fitted, 'ordinal_encoders.pkl')
print('  ✓ ordinal_encoders.pkl')

# Simpan threshold kelas
joblib.dump({'low_threshold': low_threshold, 'high_threshold': high_threshold},
            'target_quantiles.pkl')
print('  ✓ target_quantiles.pkl')

# Simpan preprocessing info
preprocessing_info = {
    'numeric_cols'      : numeric_cols,
    'ordinal_cols'      : list(ordinal_mappings.keys()),
    'nominal_cols'      : nominal_cols_exist,
    'selected_features' : list(best_result['selected_features']),
    'target_col'        : target_col,
    'feature_engineering': {}   # tidak digunakan
}
joblib.dump(preprocessing_info, 'preprocessing_info.pkl')
print('  ✓ preprocessing_info.pkl')

print('\nSemua artefak berhasil disimpan!')



Menyimpan model terbaik dan transformer...
  ✓ model_rf_3class.pkl
  ✓ onehot_encoder.pkl
  ✓ ordinal_encoders.pkl
  ✓ target_quantiles.pkl
  ✓ preprocessing_info.pkl

Semua artefak berhasil disimpan!


In [9]:
# ============================================================
# LANGKAH 8: VISUALISASI GRAFIK LENGKAP KUALITAS JURNAL
# ============================================================
print('\n' + '='*80)
print('LANGKAH 8: MEMBUAT GRAFIK EVALUASI LENGKAP (STANDAR JURNAL)')
print('='*80)

from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
from sklearn.model_selection import train_test_split
from itertools import cycle

# Menggunakan komponen model terbaik yang sudah didapat
model_best  = best_result['model']
scaler_best = best_result['scaler']
sel_feats   = best_result['selected_features']
classes_out = list(model_best.classes_)

# ── 0. Siapkan Data Test Representatif (20% Hold-out) ──
X_train_v, X_test_v, y_train_cls, y_test_cls = train_test_split(
    X[sel_feats], y_3class, test_size=0.20, random_state=RANDOM_STATE, stratify=y_3class
)
# Split untuk regresi (indeks disamakan)
_, _, y_train_reg, y_test_reg = train_test_split(
    X[sel_feats], y_reg, test_size=0.20, random_state=RANDOM_STATE
)

# Terapkan scaler jika model terbaik menggunakannya
if scaler_best:
    X_train_v = scaler_best.transform(X_train_v)
    X_test_v  = scaler_best.transform(X_test_v)
else:
    X_train_v = X_train_v.values
    X_test_v  = X_test_v.values

# Prediksi Klasifikasi
y_pred_cls = model_best.predict(X_test_v)
y_prob_cls = model_best.predict_proba(X_test_v)

# Latih ulang RF Regressor singkat untuk grafik regresi
rf_reg_vis = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE)
rf_reg_vis.fit(X_train_v, y_train_reg)
y_pred_reg = rf_reg_vis.predict(X_test_v)

sns.set_theme(style='whitegrid')
plt.rcParams['font.family'] = 'sans-serif'

# ── 1. GRAFIK INTERPRETASI MODEL ──
print('Membuat Grafik Interpretasi Model...')
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1a. Feature Importance Plot (Bar Chart)
importances = model_best.feature_importances_
idx_sort = np.argsort(importances)
axes[0].barh(range(len(idx_sort)), importances[idx_sort], color='#4C72B0', edgecolor='black')
axes[0].set_yticks(range(len(idx_sort)))
axes[0].set_yticklabels(np.array(sel_feats)[idx_sort])
axes[0].set_title('Random Forest Feature Importance', fontweight='bold')
axes[0].set_xlabel('Gini Importance')

# 1b. SHAP Summary Plot (Dot Plot)
# Catatan: Untuk multiclass, kita ambil plot untuk kelas target (misal index 0 / Beresiko) agar jadi dot plot
explainer = shap.TreeExplainer(model_best)
shap_vals = explainer.shap_values(X_test_v)
class_target_idx = 0 # Ubah jika ingin menyorot kelas lain
shap_target = shap_vals[class_target_idx] if isinstance(shap_vals, list) else (
              shap_vals[:, :, class_target_idx] if len(shap_vals.shape) == 3 else shap_vals)

plt.sca(axes[1])
shap.summary_plot(shap_target, X_test_v, feature_names=sel_feats, show=False)
axes[1].set_title(f'SHAP Summary Plot (Kelas: {classes_out[class_target_idx]})', fontweight='bold')

plt.tight_layout()
plt.savefig('8_1_Interpretasi_Model.png', dpi=300, bbox_inches='tight')
plt.close()

# ── 2. GRAFIK EVALUASI KINERJA KLASIFIKASI ──
print('Membuat Grafik Evaluasi Klasifikasi...')
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 2a. Confusion Matrix (Heatmap)
cm = confusion_matrix(y_test_cls, y_pred_cls, labels=classes_out)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0], 
            xticklabels=classes_out, yticklabels=classes_out, cbar=False)
axes[0].set_title('Confusion Matrix', fontweight='bold')
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('True Label')

# 2b. ROC Curve (Multiclass OVR)
y_test_bin = label_binarize(y_test_cls, classes=classes_out)
n_classes = y_test_bin.shape[1]
fpr, tpr, roc_auc = dict(), dict(), dict()

for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_prob_cls[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

colors = cycle(['blue', 'red', 'green', 'orange'])
for i, color in zip(range(n_classes), colors):
    axes[1].plot(fpr[i], tpr[i], color=color, lw=2,
                 label=f'ROC {classes_out[i]} (AUC = {roc_auc[i]:.2f})')

axes[1].plot([0, 1], [0, 1], 'k--', lw=2)
axes[1].set_xlim([0.0, 1.05])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('Multiclass ROC Curve', fontweight='bold')
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.savefig('8_2_Kinerja_Klasifikasi.png', dpi=300, bbox_inches='tight')
plt.close()

# ── 3. GRAFIK EVALUASI KINERJA REGRESI ──
print('Membuat Grafik Evaluasi Regresi...')
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 3a. Actual vs Predicted Plot
axes[0].scatter(y_test_reg, y_pred_reg, alpha=0.6, color='#4C72B0', edgecolor='k')
min_val = min(min(y_test_reg), min(y_pred_reg))
max_val = max(max(y_test_reg), max(y_pred_reg))
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='y = x (Perfect Prediction)')
axes[0].set_title('Actual vs Predicted Output', fontweight='bold')
axes[0].set_xlabel('Nilai Asli (Actual)')
axes[0].set_ylabel('Prediksi Model (Predicted)')
axes[0].legend()

# 3b. Residual Plot
residuals = y_test_reg - y_pred_reg
axes[1].scatter(y_pred_reg, residuals, alpha=0.6, color='#C44E52', edgecolor='k')
axes[1].axhline(y=0, color='r', linestyle='--', lw=2)
axes[1].set_title('Residual Plot (Distribusi Error)', fontweight='bold')
axes[1].set_xlabel('Prediksi Model (Predicted)')
axes[1].set_ylabel('Selisih / Error (Actual - Predicted)')

plt.tight_layout()
plt.savefig('8_3_Kinerja_Regresi.png', dpi=300, bbox_inches='tight')
plt.close()

# ── 4. GRAFIK VALIDASI PARAMETER (OOB ERROR) ──
print('Membuat Grafik OOB Error...')
min_estimators = 15
max_estimators = 150
error_rate = []

# Iterasi penambahan pohon untuk mengecek stabilitas model
for i in range(min_estimators, max_estimators + 1, 5):
    rf_oob = RandomForestClassifier(n_estimators=i, oob_score=True, random_state=RANDOM_STATE, 
                                    class_weight='balanced', n_jobs=-1)
    rf_oob.fit(X_train_v, y_train_cls)
    oob_error = 1 - rf_oob.oob_score_
    error_rate.append((i, oob_error))

trees, errors = zip(*error_rate)

plt.figure(figsize=(8, 5))
plt.plot(trees, errors, marker='o', color='#55A868', linestyle='-', lw=2)
plt.title('OOB Error Rate vs Number of Trees', fontweight='bold')
plt.xlabel('Jumlah Pohon (n_estimators)')
plt.ylabel('OOB Error Rate')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig('8_4_OOB_Error_Validation.png', dpi=300)
plt.close()

print('✓ Selesai! Semua grafik standar jurnal berhasil di-generate dan disimpan (PNG).')


LANGKAH 8: MEMBUAT GRAFIK EVALUASI LENGKAP (STANDAR JURNAL)
Membuat Grafik Interpretasi Model...
Membuat Grafik Evaluasi Klasifikasi...
Membuat Grafik Evaluasi Regresi...
Membuat Grafik OOB Error...
✓ Selesai! Semua grafik standar jurnal berhasil di-generate dan disimpan (PNG).
